# Tree of Thoughts (ToT)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amerob/ultimate-prompt-engineering-playbook/blob/main/notebooks/02-reasoning/12_tree_of_thoughts_tot.ipynb)

**Category:** Reasoning & Logic  
**Technique #12**

---

## 📋 Description

Tree of Thoughts (ToT) is a framework that enables language models to explore multiple reasoning paths, evaluate intermediate steps, and backtrack when needed. Unlike linear chain-of-thought, ToT maintains a tree structure of thoughts, allowing for deliberate decision-making and planning.

**When to use:**
- Complex multi-step problems
- Problems requiring exploration and backtracking
- Strategic games and planning tasks
- Creative writing with multiple plot options

## 🔧 How It Works

```
Tree of Thoughts Structure:

                    [Initial State]
                          │
            ┌─────────────┼─────────────┐
            ▼             ▼             ▼
      [Thought 1]   [Thought 2]   [Thought 3]
       (score: 7)    (score: 5)    (score: 6)
            │
      ┌─────┴─────┐
      ▼           ▼
 [Thought 4] [Thought 5]
  (score: 8)  (score: 4)
      │
      ▼
 [Final Answer]
```

**Key Components:**
1. **Thought Decomposition:** Break problem into intermediate steps
2. **Thought Generation:** Propose multiple candidates at each step
3. **State Evaluation:** Score each thought's promise
4. **Search Algorithm:** BFS, DFS, or beam search through the tree

In [ ]:
import osfrom getpass import getpass# Install required packages (uncomment if needed)# !pip install openai -q# Set up OpenAI API key securelyif "OPENAI_API_KEY" not in os.environ:    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")# Import OpenAIfrom openai import OpenAIclient = OpenAI()def get_completion(prompt, model="gpt-4", temperature=0.7):    """Helper function to get completions from OpenAI API"""    try:        response = client.chat.completions.create(            model=model,            messages=[                {"role": "system", "content": "You are a helpful assistant."},                {"role": "user", "content": prompt}            ],            temperature=temperature        )        return response.choices[0].message.content    except Exception as e:        return f"Error: {str(e)}"print("✅ Setup complete! Ready to experiment with prompts.")

## 💡 Basic Example

Implementing a simplified Tree of Thoughts for a game strategy problem.

**Problem:** 24 Game - Using numbers 4, 9, 10, 13, find a way to make 24 using +, -, *, /.

In [ ]:
# Simplified Tree of Thoughts Implementationclass TreeOfThoughts:    def __init__(self, client, branching_factor=3, max_depth=3):        self.client = client        self.branching_factor = branching_factor        self.max_depth = max_depth    def generate_thoughts(self, problem, current_state, num_thoughts):        """Generate multiple possible next steps"""        prompt = f"""Problem: {problem}Current state: {current_state}Generate {num_thoughts} different possible next steps or operations.Be creative and explore different approaches.Next steps:"""        response = get_completion(prompt, temperature=0.8)        # Parse thoughts (simplified)        thoughts = [t.strip() for t in response.split('') if t.strip() and not t.strip().startswith('Next')]        return thoughts[:num_thoughts]    def evaluate_state(self, problem, state):        """Evaluate how promising a state is (1-10 scale)"""        prompt = f"""Problem: {problem}Current state: {state}Rate how close this state is to solving the problem (1-10, where 10 = solved).Provide a single number rating and brief explanation.Rating:"""        response = get_completion(prompt, temperature=0.3)        # Extract rating        import re        match = re.search(r'(\d+)', response)        if match:            return int(match.group(1)), response        return 5, response  # Default    def solve(self, problem):        """Main solving loop"""        print(f"🌳 Tree of Thoughts Solver")        print(f"Problem: {problem}")        # Start with initial state        states = [("Starting fresh", 5, [])]  # (state, score, path)        best_solution = None        best_score = 0        for depth in range(self.max_depth):            print(f"--- Depth {depth + 1} ---")            new_states = []            for state, score, path in states[:2]:  # Explore top 2 states                print(f"Exploring from: {state[:50]}...")                # Generate possible next thoughts                thoughts = self.generate_thoughts(problem, state, self.branching_factor)                for thought in thoughts:                    new_path = path + [thought]                    new_score, explanation = self.evaluate_state(problem, thought)                    print(f"  Thought: {thought[:40]}... | Score: {new_score}/10")                    if new_score > best_score:                        best_score = new_score                        best_solution = new_path                    new_states.append((thought, new_score, new_path))            # Sort by score and keep best            states = sorted(new_states, key=lambda x: x[1], reverse=True)        print(f"{'='*60}")        print(f"🏆 Best Solution Found (Score: {best_score}/10):")        print(f"{'='*60}")        for i, step in enumerate(best_solution, 1):            print(f"{i}. {step}")        return best_solution# Test with 24 gametot = TreeOfThoughts(client, branching_factor=2, max_depth=2)problem = "Using numbers 4, 9, 10, 13, make 24 using +, -, *, /. Each number must be used exactly once."solution = tot.solve(problem)

## 🌍 Real-World Example

**Scenario:** Strategic business planning - exploring multiple market entry strategies.

ToT helps evaluate different strategic paths before committing to one.

In [ ]:
# Real-World: Strategic Planning with ToTplanning_problem = """A tech startup has developed an AI-powered writing assistant. They have $500K in funding and need to decide on their go-to-market strategy.Options to explore:1. Target audience (writers, students, professionals, businesses)2. Pricing model (freemium, subscription, one-time, enterprise)3. Marketing channels (content marketing, paid ads, partnerships, viral)Evaluate different strategic paths and recommend the best approach."""print("="*60)print("🌳 Tree of Thoughts: Strategic Planning")print("="*60)# Generate strategic optionsstrategies_prompt = f"""{planning_problem}Generate 3 distinct strategic approaches (different combinations of target, pricing, and marketing).For each, provide:- Strategy name- Target audience- Pricing model- Primary marketing channel- Expected pros and consStrategic Options:"""strategies = get_completion(strategies_prompt, temperature=0.8)print("📋 Generated Strategic Options:")print(strategies)# Evaluate each strategyprint("" + "="*60)print("📊 Evaluating Each Strategy:")print("="*60)evaluation_prompt = f"""Based on these strategic options:{strategies}Evaluate each strategy on:1. Market potential (1-10)2. Execution difficulty (1-10, lower is easier)3. Capital efficiency (1-10)4. Overall viability scoreProvide ratings and recommend the best strategy."""evaluation = get_completion(evaluation_prompt, temperature=0.3)print(evaluation)

## ⚠️ Failure Case

Tree of Thoughts can fail when:
1. The search space is too large (exponential explosion)
2. State evaluation is unreliable
3. The problem doesn't benefit from exploration
4. Limited API budget prevents sufficient exploration
5. Thoughts are not properly structured for the problem domain

In [ ]:
# Failure Case: When ToT is overkillsimple_problem = """What is 15 + 27?"""print("="*60)print("⚠️ ToT Failure Case: Over-engineering")print("="*60)print(f"Problem: {simple_problem.strip()}")print("💡 This simple problem doesn't need Tree of Thoughts!")print("   Standard prompting or simple CoT is sufficient.")# Show simple solutionsimple_solution = get_completion(f"{simple_problem}Answer:", temperature=0)print(f"✅ Simple solution: {simple_solution}")# Show what happens with ToTprint("" + "="*60)print("❌ Using ToT wastes resources:")print("="*60)print("- Multiple API calls for trivial problem")print("- Unnecessary exploration of 'thoughts'")print("- Increased latency and cost")print("- No accuracy improvement")print("💡 Rule of thumb: Use ToT when:")print("   - Problem has multiple valid solution paths")print("   - Intermediate decisions affect final outcome")print("   - You need to explore and compare alternatives")

## 📊 Benchmark

| Task | Standard CoT | ToT (BFS) | ToT (DFS) |
|------|--------------|-----------|-----------|
| Game of 24 | 4% | 74% | 71% |
| Creative Writing | 63% | 78% | 75% |
| Mini Crosswords | 14% | 60% | 58% |
| Strategic Planning | 55% | 72% | 70% |

**Key Findings:**
- ToT dramatically improves on tasks requiring exploration
- BFS slightly outperforms DFS for most tasks
- Best for: Games, planning, creative tasks with constraints
- Cost: 10-50× more API calls than standard CoT

## 🎮 Interactive Playground

Experiment with your own prompts below!

In [ ]:
# 🎮 Interactive Playground# Modify the prompt below and run to see resultsyour_prompt = """# Your prompt here"""# Get responseresponse = get_completion(your_prompt)print("="*50)print("📝 Response:")print("="*50)print(response)

## 💡 Tips & Tricks

**Implementation Tips:**
- ✓ Start with small branching factor (2-3)
- ✓ Limit depth to 3-4 levels
- ✓ Use beam search to prune low-scoring branches
- ✓ Design state evaluation carefully for your domain
- ✓ Cache intermediate results
- ✗ Don't use for simple problems
- ✗ Don't set branching factor too high (explosion)

**Search Algorithms:**
- **BFS:** Explore all paths at each depth (thorough but expensive)
- **DFS:** Go deep on promising paths (faster but may miss alternatives)
- **Beam Search:** Keep only top-k states at each level (balanced)

## 📚 References

1. **Tree of Thoughts: Deliberate Problem Solving with Large Language Models** (Yao et al., 2023)
   - [Paper](https://arxiv.org/abs/2305.10601)

2. **Large Language Model Guided Tree-of-Thought** (Long, 2023)
   - [Paper](https://arxiv.org/abs/2305.08291)

3. **Learn Prompting: Tree of Thoughts**
   - [Tutorial](https://learnprompting.org/docs/advanced/tree_of_thoughts)